In [0]:
dbutils.widgets.text("catalog", "weather_dev", "Catálogo de destino")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, DoubleType

# Leemos la tabla bronze completa
df_bronze = spark.table(f"{catalog}.bronze.weather_forecast_raw")

# Definimos la forma del JSON dentro de raw_response (solo la parte "hourly" que nos interesa)
hourly_schema = StructType([
    StructField("time", ArrayType(StringType())),
    StructField("temperature_2m", ArrayType(DoubleType())),
    StructField("precipitation", ArrayType(DoubleType())),
    StructField("wind_speed_10m", ArrayType(DoubleType())),
])
raw_schema = StructType([StructField("hourly", hourly_schema)])

# Parseamos el JSON y "desempacamos" los 4 arreglos juntos, hora por hora
df_silver = (
    df_bronze
    .withColumn("parsed", F.from_json(F.col("raw_response"), raw_schema))
    .withColumn("zipped", F.arrays_zip(
        F.col("parsed.hourly.time"),
        F.col("parsed.hourly.temperature_2m"),
        F.col("parsed.hourly.precipitation"),
        F.col("parsed.hourly.wind_speed_10m"),
    ))
    .withColumn("hour_data", F.explode("zipped"))
    .select(
        F.col("hour_data.time").cast("timestamp").alias("forecast_time"),
        F.col("hour_data.temperature_2m").alias("temperature_c"),
        F.col("hour_data.precipitation").alias("precipitation_mm"),
        F.col("hour_data.wind_speed_10m").alias("wind_speed_kmh"),
        "latitude",
        "longitude",
        "ingestion_timestamp",
        "ingestion_date",
    )
    .withColumn("forecast_date", F.to_date("forecast_time"))
)

display(df_silver)

In [0]:
table_name = f"{catalog}.silver.weather_forecast_hourly"

(df_silver.write
    .format("delta")
    .mode("append")
    .partitionBy("forecast_date")
    .saveAsTable(table_name)
)

display(spark.sql(f"SELECT * FROM {table_name} ORDER BY forecast_time LIMIT 10"))